In [1]:
import pandas as pd
df=pd.read_csv("DateFruit_Dataset.csv")
df.sample()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
785,361867,2681.0071,909.2697,510.2585,0.8277,678.7808,0.9594,377190,0.6609,1.782,...,14.5366,14.03,8.4396,-8114013696,-11200927744,-13146814464,23.4922,28.5482,31.726,SAFAVI


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
x=df.drop("Class",axis=1)
y=df["Class"].copy()
le=LabelEncoder()
y=le.fit_transform(y)
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [4]:
scaler=StandardScaler()
x_trains=scaler.fit_transform(x_train)
x_tests=scaler.transform(x_test)

# ANN
 

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

In [6]:
#creating torchs
##traing
x_train_tensor=torch.tensor(x_trains,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.long)
##validation
x_test_tensor=torch.tensor(x_tests,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test,dtype=torch.long)

In [9]:
#TensorDataset
train_dataset=TensorDataset(x_train_tensor,y_train_tensor)
test_dataset=TensorDataset(x_test_tensor,y_test_tensor)

In [10]:
#DataLoader
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=True)

In [14]:
#model
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        self.model=nn.Sequential(
            nn.Linear(x.shape[1],64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,7)
        )
    def forward(self,x):
        return self.model(x)
            

In [17]:
model=ANN()
#loss
criteria=nn.CrossEntropyLoss()
#optimizer
optimizer=optim.Adam(model.parameters())

In [19]:
#training
epochs=50
for epoch in range(epochs):
    model.train()
    running_loss=0.0
    for xb,yb in train_loader:
        optimizer.zero_grad()
        outputs=model(xb)
        loss=criteria(outputs,yb)
        loss.backward()
        optimizer.step()
        running_loss+=loss.item()
    train_loss=running_loss/len(train_loader)
    print(f"{epoch}/{epochs} traing loss:{train_loss} ")
        
        
    

0/50 traing loss:1.6584624103877856 
1/50 traing loss:1.0408468842506409 
2/50 traing loss:0.7024506330490112 
3/50 traing loss:0.5529372381127399 
4/50 traing loss:0.4607328733672266 
5/50 traing loss:0.4027370940084043 
6/50 traing loss:0.3609610012044077 
7/50 traing loss:0.32977688766044116 
8/50 traing loss:0.30525017918451974 
9/50 traing loss:0.2794389783040337 
10/50 traing loss:0.25241992460644763 
11/50 traing loss:0.23961344361305237 
12/50 traing loss:0.22237783638031586 
13/50 traing loss:0.21772985970196518 
14/50 traing loss:0.20136133387036945 
15/50 traing loss:0.1991023729028909 
16/50 traing loss:0.18697161859144334 
17/50 traing loss:0.17892437525417493 
18/50 traing loss:0.18091768253108728 
19/50 traing loss:0.17308367788791656 
20/50 traing loss:0.1638480091224546 
21/50 traing loss:0.16190952418938928 
22/50 traing loss:0.15588440227767694 
23/50 traing loss:0.1571990905896477 
24/50 traing loss:0.15750102285781634 
25/50 traing loss:0.15331700703372125 
26/50 t

In [24]:
#Evaluation
model.eval()
total=0
crt=0
with torch.no_grad():
    for xb,yb in test_loader:
        outputs=model(xb)
        _,pred=torch.max(outputs,1)
        crt+=(pred==yb).sum().item()
        total+=yb.size(0)
print("total:",total)
print("crt_pred:",crt)
print("Accuracy:",crt/total*100)

total: 180
crt_pred: 172
Accuracy: 95.55555555555556
